# Gaussian Processes

Rasmussen & Williams (2006): http://www.gaussianprocess.org/gpml/




Sparse GPs (2009): http://proceedings.mlr.press/v5/titsias09a.html






Deep GPs (2013): https://arxiv.org/abs/1211.0358







Neural Tangent Kernel (2018): https://arxiv.org/abs/1806.07572

# Gaussian Process Classifier

In [1]:
# Arabic Sentiment Analysis — Gaussian Process Classifier (Rasmussen & Williams, 2006)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.decomposition import TruncatedSVD

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, Matern
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_clean.csv"
VAL_PATH      = "/content/val_clean.csv"
TEST_PATH     = "/content/test_clean.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

LSA_DIM       = 50     # GP is O(n³): keep dimensions very low
MAX_TRAIN_GP  = 3_000  # subsample training data for tractability

TFIDF_PARAMS = dict(
    analyzer="word", ngram_range=(1,3), max_features=50_000,
    sublinear_tf=True, min_df=2, max_df=0.95,
    strip_accents=None, token_pattern=r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING
print("=" * 60)
print("  ⚙️  Building Features (TF-IDF → LSA)")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)
X_tr_t = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_va_t = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_te_t = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

lsa = TruncatedSVD(n_components=LSA_DIM, random_state=RANDOM_STATE)
X_tr_lsa = lsa.fit_transform(hstack([X_tr_t, emoji_sparse(train_df)]))
X_va_lsa = lsa.transform(hstack([X_va_t, emoji_sparse(val_df)]))
X_te_lsa = lsa.transform(hstack([X_te_t, emoji_sparse(test_df)]))

scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_tr_lsa)
X_val        = scaler.transform(X_va_lsa)
X_test       = scaler.transform(X_te_lsa)
print(f"Feature shape (train full) : {X_train_full.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train_full = le.fit_transform(train_df[LABEL_COL])
y_val        = le.transform(val_df[LABEL_COL])
y_test       = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. SUBSAMPLE FOR TRACTABILITY
if len(train_df) > MAX_TRAIN_GP:
    rng  = np.random.default_rng(RANDOM_STATE)
    idx  = rng.choice(len(train_df), size=MAX_TRAIN_GP, replace=False)
    X_train  = X_train_full[idx]
    y_train  = y_train_full[idx]
    print(f"⚠️  GP is O(n³): subsampled {MAX_TRAIN_GP:,} of {len(train_df):,} training points.\n")
else:
    X_train, y_train = X_train_full, y_train_full

# 5. MODEL — Gaussian Process Classifier (RBF kernel, Laplace approximation)
# Kernel: C(1.0) * RBF(1.0) — constant amplitude × radial basis function
# n_restarts_optimizer: re-starts gradient-based kernel hyperparameter optimisation
# warm_starts: reuses previous Laplace approximation for each optimiser restart
print("=" * 60)
print("  🔮 Training: Gaussian Process Classifier (Rasmussen & Williams, 2006)")
print("=" * 60)

kernel = C(1.0, (1e-2, 1e2)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))

clf = GaussianProcessClassifier(
    kernel                = kernel,
    n_restarts_optimizer  = 3,
    max_iter_predict      = 200,
    warm_start            = True,
    multi_class           = "one_vs_rest",
    n_jobs                = -1,
    random_state          = RANDOM_STATE,
)
clf.fit(X_train, y_train)
print(f"\n  Optimised kernel : {clf.kernel_}\n")

# 6. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  GP Classifier  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features (TF-IDF → LSA)
Feature shape (train full) : (5067, 50)

Classes : [0 1]  →  [0, 1]

⚠️  GP is O(n³): subsampled 3,000 of 5,067 training points.

  🔮 Training: Gaussian Process Classifier (Rasmussen & Williams, 2006)

  Optimised kernel : 4.17**2 * RBF(length_scale=11.1)


────────────────────────────────────────────────────────────
  GP Classifier  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7422
  Precision : 0.7439
  Recall    : 0.7422
  F1-Score  : 0.7417

  Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.78      0.75       543
           1       0.76      0.70      0.73       543

    accuracy                           0.74      1086
   macro avg       0.74      0.74      0.74      1086
weighted avg       0.